# 03 — Base dataset and temporal splits

Build the modeling population (`base_dataset`) and define train/val/test cuts aligned to **week-ending Tuesdays**.

**Prediction contract:** predict `rpm` for week-ending Tuesday `t` using only information with `date ≤ t−1`, plus ex-ante features (OD, distance, calendar).

**History window:** prefer a recent snapshot (`date ≥ 2024-07-01` from the download script). Walk-forward forecasts start in **2025**.

Prerequisites: `python scripts/download_usda_data.py` (default `--start-date 2024-07-01`).


## 1. Contract and week-ending Tuesday

FVWTRK labels each report as *week ending Tuesday*. Example: `date = 2026-08-18` covers roughly **Aug 12–18**.

| Name | Meaning |
|---|---|
| `raw_dataset` | USDA API snapshot (optionally date-filtered at download) |
| `base_dataset` | Lane-week panel for modeling/scoring |

| Config | Value | Role |
|---|---|---|
| API / raw `start_date` | `2024-07-01` | Burn-in before 2025 forecasts |
| First walk-forward forecast | first Tuesday ≥ `2025-01-01` | Official eval start |


## 2. Build `base_dataset`


In [8]:
from pathlib import Path

import numpy as np
import pandas as pd

from freight_rates.ingestion import (
    DEFAULT_START_DATE,
    load_raw_metadata,
    load_raw_snapshot,
)
from freight_rates.preprocessing import build_lane_week_panel

ROOT = Path("..").resolve()
RAW_DIR = ROOT / "data" / "raw"

# Modeling window (aligns with default download filter)
API_START_DATE = pd.Timestamp(DEFAULT_START_DATE)  # 2024-07-01
FIRST_FORECAST_DATE = pd.Timestamp("2025-01-07")   # first week-ending Tuesday in 2025

raw_dataset = load_raw_snapshot(raw_dir=RAW_DIR)
try:
    raw_meta = load_raw_metadata(raw_dir=RAW_DIR)
except FileNotFoundError:
    raw_meta = {}

print("Snapshot metadata:")
print(f"  query_start_date: {raw_meta.get('query_start_date')}")
print(f"  query_end_date:   {raw_meta.get('query_end_date')}")
print(f"  min_date:         {raw_meta.get('min_date')}")
print(f"  max_date:         {raw_meta.get('max_date')}")
print(f"  row_count:        {raw_meta.get('row_count')}")

base_full = build_lane_week_panel(raw_dataset)

# Enforce modeling window even if an older full-history parquet is present
base_dataset = base_full.loc[base_full["date"] >= API_START_DATE].copy().reset_index(drop=True)

summary = pd.Series(
    {
        "raw_rows": len(raw_dataset),
        "base_rows_full_snapshot": len(base_full),
        "base_rows_model_window": len(base_dataset),
        "api_start_date": API_START_DATE.date().isoformat(),
        "date_min": base_dataset["date"].min().date().isoformat(),
        "date_max": base_dataset["date"].max().date().isoformat(),
        "n_unique_dates": base_dataset["date"].nunique(),
        "n_lanes": base_dataset["lane_id"].nunique(),
        "n_origins": base_dataset["origin"].nunique(),
        "n_destinations": base_dataset["destination"].nunique(),
        "rpm_conflict_pct": 100.0 * base_dataset["rpm_conflict"].mean(),
        "first_forecast_date": FIRST_FORECAST_DATE.date().isoformat(),
    }
)
summary.to_frame("value")


Snapshot metadata:
  query_start_date: 2024-07-01
  query_end_date:   None
  min_date:         2024-07-02
  max_date:         2026-08-18
  row_count:        9043


,value
raw_rows,9043
base_rows_full_snapshot,8202
base_rows_model_window,8202
api_start_date,2024-07-01
date_min,2024-07-02
date_max,2026-08-18
n_unique_dates,110
n_lanes,188
n_origins,31
n_destinations,10


## 3. Calendar QA

Confirm `date` behaves as a weekly Tuesday index inside the modeling window.


In [9]:
dates = base_dataset["date"]
unique_dates = dates.drop_duplicates().sort_values().reset_index(drop=True)

dow = dates.dt.day_name().value_counts(normalize=True).mul(100).round(2)
gap_days = unique_dates.diff().dt.days.dropna()

calendar_qa = pd.Series(
    {
        "pct_rows_tuesday": 100.0 * (dates.dt.dayofweek == 1).mean(),
        "pct_unique_dates_tuesday": 100.0 * (unique_dates.dt.dayofweek == 1).mean(),
        "median_gap_days": gap_days.median(),
        "pct_gaps_eq_7": 100.0 * (gap_days == 7).mean(),
        "n_gaps_ne_7": int((gap_days != 7).sum()),
    }
)
print("Day-of-week share of rows (%):")
display(dow.to_frame("pct_rows"))
print("Gap distribution between consecutive unique dates (days):")
display(gap_days.value_counts().sort_index().to_frame("n_gaps"))
calendar_qa.to_frame("value")


Day-of-week share of rows (%):


,pct_rows
date,
Tuesday,100.0


Gap distribution between consecutive unique dates (days):


,n_gaps
date,
7.0,107
14.0,2


,value
pct_rows_tuesday,100.000000
pct_unique_dates_tuesday,100.000000
median_gap_days,7.000000
pct_gaps_eq_7,98.165138
n_gaps_ne_7,2.000000


In [10]:
gap_table = pd.DataFrame(
    {
        "prev_date": unique_dates.iloc[:-1].to_numpy(),
        "next_date": unique_dates.iloc[1:].to_numpy(),
        "gap_days": gap_days.to_numpy(),
    }
)
irregular = gap_table.loc[gap_table["gap_days"] != 7].sort_values("gap_days", ascending=False)
print(f"Irregular gaps: {len(irregular):,}")
irregular.head(20)


Irregular gaps: 2


,prev_date,next_date,gap_days
24,2024-12-17,2024-12-31,14.0
75,2025-12-16,2025-12-30,14.0


**Takeaway:** Report dates are overwhelmingly week-ending Tuesdays with 7-day spacing. Keep split cutoffs on Tuesdays present in the panel.


## 4. Temporal splits (diagnostic / illustrative only)

This section is **not** the official evaluation protocol. It only sketches a fixed train/val/test layout for diagnostics (lane overlap, size checks). **Headline metrics use walk-forward** (Section 5).

`train` uses **`date ≤ 2024-12-31`**, so it intentionally includes the full burn-in interval from the modeling-window start through that Tuesday — e.g. **2024-07-02 → 2024-12-31**, not a single week. Dates before `2024-12-31` in train are expected.

| Split | Rule | Role |
|---|---|---|
| `train` | `API_START ≤ date ≤ 2024-12-31` | 2024 H2 burn-in (history seed) |
| `val` | `2025-01-07 ≤ date ≤ 2025-06-24` | early-2025 tuning (optional) |
| `test` | `date ≥ 2025-07-01` | late holdout (optional) |


In [11]:
TRAIN_END = pd.Timestamp("2024-12-31")    # last week-ending Tuesday in 2024
VAL_START = FIRST_FORECAST_DATE           # 2025-01-07
VAL_END = pd.Timestamp("2025-06-24")
TEST_START = pd.Timestamp("2025-07-01")


def assign_temporal_split(date: pd.Series) -> pd.Series:
    """Map week-ending Tuesday dates to train/val/test."""
    out = pd.Series(index=date.index, dtype="object")
    out[date <= TRAIN_END] = "train"
    out[(date >= VAL_START) & (date <= VAL_END)] = "val"
    out[date >= TEST_START] = "test"
    return out


panel = base_dataset.copy()
panel["split"] = assign_temporal_split(panel["date"])

unassigned = panel.loc[panel["split"].isna(), "date"].drop_duplicates()
assert panel["split"].notna().all(), f"Unassigned dates: {unassigned.tolist()}"

split_summary = (
    panel.groupby("split", sort=False)
    .agg(
        n_rows=("rpm", "size"),
        n_lanes=("lane_id", "nunique"),
        date_min=("date", "min"),
        date_max=("date", "max"),
        median_rpm=("rpm", "median"),
    )
    .reindex(["train", "val", "test"])
)
split_summary["date_min"] = split_summary["date_min"].dt.date
split_summary["date_max"] = split_summary["date_max"].dt.date
split_summary


,n_rows,n_lanes,date_min,date_max,median_rpm
split,,,,,
train,2513,139,2024-07-02,2024-12-31,2.775862
val,1997,117,2025-01-07,2025-06-24,2.767857
test,3692,146,2025-07-01,2026-08-18,3.153846


In [12]:
lanes = {
    s: set(panel.loc[panel["split"] == s, "lane_id"].unique())
    for s in ("train", "val", "test")
}
overlap = pd.Series(
    {
        "lanes_train": len(lanes["train"]),
        "lanes_val": len(lanes["val"]),
        "lanes_test": len(lanes["test"]),
        "val_also_in_train": len(lanes["val"] & lanes["train"]),
        "test_also_in_train": len(lanes["test"] & lanes["train"]),
        "test_never_in_train": len(lanes["test"] - lanes["train"]),
        "pct_test_lanes_new": 100.0
        * len(lanes["test"] - lanes["train"])
        / max(len(lanes["test"]), 1),
    }
)
overlap.to_frame("value")


,value
lanes_train,139.000000
lanes_val,117.000000
lanes_test,146.000000
val_also_in_train,99.000000
test_also_in_train,98.000000
test_never_in_train,48.000000
pct_test_lanes_new,32.876712


In [13]:
obs = (
    panel.groupby(["split", "lane_id"])
    .size()
    .rename("n_obs")
    .reset_index()
)
obs_stats = (
    obs.groupby("split")["n_obs"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9])
    .reindex(["train", "val", "test"])
)
obs_stats


,count,mean,std,min,25%,50%,75%,90%,max
split,,,,,,,,,
train,139.0,18.079137,9.465471,1.0,8.5,26.0,26.0,26.0,26.0
val,117.0,17.068376,10.056501,1.0,4.0,24.0,25.0,25.0,25.0
test,146.0,25.287671,20.889515,1.0,5.0,17.0,43.0,59.0,59.0


**Takeaway:** Train covering 2024-07 → 2024-12 is the burn-in block (`≤ 2024-12-31`), not a bug. These fixed splits are illustrative; use Section 5 walk-forward for training/evaluation error over weeks.


## 5. Walk-forward evaluation protocol (design)

Primary evaluation: **expanding-window, one-step ahead**, starting in 2025.

```text
for each week-ending Tuesday t >= 2025-01-07:
    train on rows with date ≤ t−1   (within downloaded history)
    predict rpm for lanes observed at t
    record error for week t
```

History before `API_START_DATE` is **not** used (by design). Expanding still grows week by week from mid-2024 onward.

| Item | Choice |
|---|---|
| Raw/API lower bound | `2024-07-01` (default download) |
| First forecast `t` | `2025-01-07` |
| Window | Expanding within the snapshot |
| Baseline | `rpm_lag_1` (global mean if lane history = 0) |
| Models | Global GBM first; hierarchical Bayesian later |

Fixed train/val/test (Section 4) remains useful for diagnostics; walk-forward is the headline protocol.

No models are trained in this notebook.


In [14]:
forecast_dates = unique_dates[unique_dates >= FIRST_FORECAST_DATE].reset_index(drop=True)

wf_preview = pd.Series(
    {
        "n_forecast_weeks": len(forecast_dates),
        "first_forecast_date": forecast_dates.iloc[0].date().isoformat() if len(forecast_dates) else None,
        "last_forecast_date": forecast_dates.iloc[-1].date().isoformat() if len(forecast_dates) else None,
        "n_rows_on_first_forecast_week": int((panel["date"] == forecast_dates.iloc[0]).sum())
        if len(forecast_dates)
        else 0,
        "n_rows_on_last_forecast_week": int((panel["date"] == forecast_dates.iloc[-1]).sum())
        if len(forecast_dates)
        else 0,
        "n_burn_in_rows_before_first_forecast": int((panel["date"] < FIRST_FORECAST_DATE).sum()),
    }
)

sample_idx = list(range(0, len(forecast_dates), 8))
if len(forecast_dates):
    sample_idx.append(len(forecast_dates) - 1)
sample_idx = sorted(set(i for i in sample_idx if i >= 0))

rows = []
for i in sample_idx:
    t = forecast_dates.iloc[i]
    n_train = int((panel["date"] < t).sum())
    n_score = int((panel["date"] == t).sum())
    rows.append({"forecast_date": t.date(), "n_train_rows": n_train, "n_score_rows": n_score})

display(wf_preview.to_frame("value"))
pd.DataFrame(rows)


,value
n_forecast_weeks,84
first_forecast_date,2025-01-07
last_forecast_date,2026-08-18
n_rows_on_first_forecast_week,89
n_rows_on_last_forecast_week,73
n_burn_in_rows_before_first_forecast,2513


,forecast_date,n_train_rows,n_score_rows
0,2025-01-07,2513,89
1,2025-03-04,3161,80
2,2025-04-29,3789,80
3,2025-06-24,4426,84
4,2025-08-19,5013,62
5,2025-10-14,5457,51
6,2025-12-09,5918,60
7,2026-02-10,6357,55
8,2026-04-07,6794,57
9,2026-06-02,7334,75


## 6. Summary

| Decision | Choice |
|---|---|
| Modeling grain | `base_dataset` lane-week |
| `date` meaning | Week-ending Tuesday |
| Prediction contract | `rpm` at `t` given info through `t−1` + ex-ante features |
| Raw/API window | `date ≥ 2024-07-01` (default) |
| First walk-forward forecast | `2025-01-07` |
| Diagnostic splits | train ≤ 2024-12-31; val 2025-01-07→2025-06-24; test ≥ 2025-07-01 |

**Re-download tip:** if your local parquet still starts in 2000, run:

```bash
python scripts/download_usda_data.py --start-date 2024-07-01
```

This notebook also filters `base_dataset` to `date ≥ 2024-07-01` in memory so older snapshots remain usable.

**Next steps:** `splits.py`; `availability_lag_1`; walk-forward GBM training.
